In [0]:
df = spark.table("novacart_dev.bronze.order_items")
display(df)

In [0]:
from pyspark.sql import functions as F

df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

In [0]:
df = df.replace(["\\N", "?", "", "null", "NULL"], None)

In [0]:
string_cols = [c for c, t in df.dtypes if t == "string"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.col(col)))

In [0]:
string_cols = [c for c, t in df.dtypes if t == "string"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.lower(F.col(col))))

In [0]:
df = df.dropDuplicates(["order_item_id"])

In [0]:
df = df.withColumn("quantity", F.col("quantity").cast("int"))
df = df.withColumn("unit_price", F.col("unit_price").cast("double"))
df = df.withColumn("line_total", F.col("line_total").cast("double"))

In [0]:
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") <= 0, None).otherwise(F.col("quantity"))
)

df = df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") <= 0, None).otherwise(F.col("unit_price"))
)

In [0]:
df = df.withColumn(
    "calculated_total",
    F.col("quantity") * F.col("unit_price")
)

In [0]:
df = df.withColumn(
    "dq_line_total_mismatch",
    F.when(
        F.abs(F.col("line_total") - F.col("calculated_total")) > 0.01,
        1
    ).otherwise(0)
)

In [0]:
df = df.withColumn(
    "line_total",
    F.col("calculated_total")
)

In [0]:
orders_df = spark.table("novacart_dev.silver.slv_orders")

df = df.join(
    orders_df.select("order_id"),
    on="order_id",
    how="left"
)

df = df.withColumn(
    "dq_orphan_order",
    F.when(F.col("order_id").isNull(), 1).otherwise(0)
)

In [0]:
products_df = spark.table("novacart_dev.silver.slv_products")

df = df.join(
    products_df.select("product_id"),
    on="product_id",
    how="left"
)

df = df.withColumn(
    "dq_orphan_product",
    F.when(F.col("product_id").isNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "dq_missing_order_item_id",
    F.when(F.col("order_item_id").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_quantity",
    F.when(F.col("quantity").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_price",
    F.when(F.col("unit_price").isNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumn("load_timestamp", F.current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.silver.slv_order_items")

In [0]:
display(spark.table("novacart_dev.silver.slv_order_items"))